In [1]:
# importing relevant libraries
import pandas as pd
import numpy as np
import json
import os
from itertools import chain

# import boto3

from tqdm.notebook import tqdm_notebook
import time

import csv
import re

# display all rows & columns
#pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# show all outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# Disable warnings
import warnings
warnings.filterwarnings("ignore")

**Import data for analysis**

In [2]:
# Import data
attrition_list = 'Hopeliner Copy of Sample 2022 Attrition List.xlsx'
responses = 'TTEC Interviews (Responses).xlsx'

attrition_df = pd.read_excel(attrition_list)
response_df = pd.read_excel(responses)


**Pre-processing of Dataset**

In [3]:
# Strip column names
attrition_df.columns = [x.strip() for x in attrition_df.columns]
response_df.columns = [x.strip() for x in response_df.columns]

# Response df - pre-processing
response_df.rename(columns =  {'Read Call Opening: \nHi (First Name), thank you for making time for this short interview or chat with us. As shared in our (email/ SMS) exchange, my name is (Interviewer Name). I am from e-BI Solutions. We are working with TTEC to find ways to improve their employee experience so we are grateful that you made time for this.':'Read Call Opening',
                              'Confidentiality Notice and Consent:\nPlease note that we will treat our conversation with confidentiality and only disclose information to the extent you wish to share them with TTEC. Our goal is to help them improve the employee experience through an unbiased study and analysis of the data we gather. We will ask for your sign-off on the final notes we will include in the study. Do you agree and consent to this interview/ short chat? (Share option to record video/ audio/ transcript only)\n\n(Wait for response. Proceed with questions as appropriate)': 'Confidentiality Notice and Consent',
                              }, inplace = True)

cols = ['Stated Reason for Leaving TTEC', 'Other considerations for leaving', 'Top 3 Low lights and impact to them',
'Highlights of their stay with TTEC', 'Suggestions on redesigning the experience',
'What would make you consider returning or referring friends and family members to TTEC?',
'Any questions for e-BI or TTEC?']

for col in cols:
    response_df[col] = response_df[col].astype(str)
    
# Add a new column for all reasons for leaving
response_df['All Leaving Reasons'] = response_df['Stated Reason for Leaving TTEC'] + " " + response_df['Other considerations for leaving']

## Pre-processing Descriptive Data

In [4]:
# pip install gensim
# pip install nltk
# pip install matplotlib
# pip install advertools
# Run 'python -m nltk.downloader all' on ubuntu
# pip install typing-extensions --upgrade

In [5]:
#import modules
import nltk
import gensim
import advertools as adv
import matplotlib.pyplot
import os.path
from gensim import corpora
from gensim.models import LsiModel
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt

# Latent Dirichlet Allocation (LDA) Topic Analysis Model

**Resources:**
- [Evaluate Topic Models: Latent Dirichlet Allocation (LDA)](https://towardsdatascience.com/evaluate-topic-model-in-python-latent-dirichlet-allocation-lda-7d57484bb5d0)

- [pyLDAvis: Topic Modelling Exploration Tool That Every NLP Data Scientist Should Know](https://neptune.ai/blog/pyldavis-topic-modelling-exploration-tool-that-every-nlp-data-scientist-should-know)

In [6]:
from gensim.utils import simple_preprocess
from nltk.corpus import stopwords

# Stop words
stop_words = stopwords.words('english') + list(adv.stopwords['tagalog'])

# Functions
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
        
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

def preprocess_data_indiv(doc_set):
    """
    Input  : docuemnt list
    Purpose: preprocess text (tokenize, removing stopwords, and stemming)
    Output : preprocessed text
    """
    # initialize regex tokenizer
    tokenizer = RegexpTokenizer(r'\w+')
    # create English & Tagalog stop words list
    en_stop = set(stopwords.words('english') + list(adv.stopwords['tagalog']))
    # Create p_stemmer of class PorterStemmer
    p_stemmer = PorterStemmer()
    # list for tokenized documents in loop
    texts = []
    # loop through document list
    for i in doc_set:
        # clean and tokenize document string
        raw = i.lower()
        tokens = tokenizer.tokenize(raw)
        # remove stop words from tokens
        stopped_tokens = [i for i in tokens if not i in en_stop]
        # stem tokens
        #stemmed_tokens = [p_stemmer.stem(i) for i in stopped_tokens]
        # add tokens to list
        texts.append(stopped_tokens)
    return texts


In [7]:
# NLTK Stop words
# import nltk
# nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english') + list(adv.stopwords['tagalog'])
stop_words.extend(['from', 'subject', 're', 'edu', 'use','employee','ttec','resign'])
# Define functions for stopwords, bigrams, trigrams and lemmatization
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]
def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]
def make_trigrams(texts):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [8]:
# Create a copy of the original response dataframe
analysis_df = response_df.copy()

> 1) Remove punctuations, digits and stop words from text

In [9]:
# Remove punctuations & digits
import re
from string import digits
remove_digits = str.maketrans('', '', digits) # remove digits

for col in cols:
    analysis_df[col] = analysis_df[col].apply(lambda x: re.sub(r'[^\w\s]', '', x).translate(remove_digits).strip().lower())

In [10]:
import spacy
# !python -m spacy download en_core_web_lg 
# !python -m spacy download en_core_web_sm

In [11]:
analysis_df.columns

Index(['Timestamp', 'Email Address', 'Read Call Opening',
       'Confidentiality Notice and Consent', 'Name of former employee',
       'Oracle ID of former employee', 'Last day with TTEC (Complete Date)',
       'Stated Reason for Leaving TTEC', 'Other considerations for leaving',
       'Top 3 Low lights and impact to them',
       'Highlights of their stay with TTEC',
       'Suggestions on redesigning the experience',
       'What would make you consider returning or referring friends and family members to TTEC?',
       'Any questions for e-BI or TTEC?',
       'Closing: Thank the person for the time and trust. Show - responses. End call.',
       'How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good',
       'All Leaving Reasons'],
      dtype='object')

In [12]:
# Remove stop words
data = analysis_df['Highlights of their stay with TTEC'].to_list() # field to analyse (MANUAL INPUT)
data_words = list(sent_to_words(data)) # convert words in response to list of words

# Build the bigram and trigram models
bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100) # higher threshold fewer phrases.
trigram = gensim.models.Phrases(bigram[data_words], threshold=100)
# Faster way to get a sentence clubbed as a trigram/bigram
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)

data_words_nostops = remove_stopwords(data_words) # remove stop words

# Form Bigrams
data_words_bigrams = make_bigrams(data_words_nostops)

# Initialize spacy 'en' model, keeping only tagger component (for efficiency)
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

# Do lemmatization keeping only noun, adj, vb, adv
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

2022-12-21 09:27:28,604 | INFO | phrases.py:497 | learn_vocab | collecting all words and their counts
2022-12-21 09:27:28,605 | INFO | phrases.py:502 | learn_vocab | PROGRESS: at sentence #0, processed 0 words and 0 word types
2022-12-21 09:27:28,608 | INFO | phrases.py:525 | learn_vocab | collected 1562 word types from a corpus of 1509 words (unigram + bigrams) and 107 sentences
2022-12-21 09:27:28,608 | INFO | phrases.py:580 | add_vocab | using 1562 counts as vocab in Phrases<0 vocab, min_count=5, threshold=100, max_vocab_size=40000000>
2022-12-21 09:27:28,609 | INFO | phrases.py:497 | learn_vocab | collecting all words and their counts
2022-12-21 09:27:28,610 | INFO | phrases.py:502 | learn_vocab | PROGRESS: at sentence #0, processed 0 words and 0 word types
2022-12-21 09:27:28,620 | INFO | phrases.py:525 | learn_vocab | collected 1562 word types from a corpus of 1509 words (unigram + bigrams) and 107 sentences
2022-12-21 09:27:28,620 | INFO | phrases.py:580 | add_vocab | using 1562

> 2) Create a corpus

In [13]:
import gensim.corpora as corpora

def prepare_corpus(doc_clean):
    """
    Input  : clean document
    Purpose: create term dictionary of our courpus and Converting list of documents (corpus) into Document Term Matrix
    Output : term dictionary and Document Term Matrix
    """
    # Creating the term dictionary of our courpus, where every unique term is assigned an index. dictionary = corpora.Dictionary(doc_clean)
    dictionary = corpora.Dictionary(doc_clean)
    # Converting list of documents (corpus) into Document Term Matrix using dictionary prepared above.
    doc_term_matrix = [dictionary.doc2bow(doc) for doc in doc_clean]
    # generate LDA model
    return dictionary,doc_term_matrix

In [14]:
import gensim.corpora as corpora
# Create Dictionary
id2word = corpora.Dictionary(data_lemmatized)
# Create Corpus
texts = data_lemmatized
# Term Document Frequency
corpus = [id2word.doc2bow(text) for text in texts]

2022-12-21 09:27:29,559 | INFO | dictionary.py:209 | add_documents | adding document #0 to Dictionary(0 unique tokens: [])
2022-12-21 09:27:29,561 | INFO | dictionary.py:214 | add_documents | built Dictionary(315 unique tokens: ['salary', 'approachable', 'atmospherend', 'career', 'compare']...) from 107 documents (total 832 corpus positions)


> 3) LDA Model 

In [15]:
from pprint import pprint

In [16]:
# number of topics
num_topics = 10

# Build LDA model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                       id2word=id2word,
                                       num_topics=num_topics,
                                      random_state=3,
                                      chunksize=100,
                                       passes=10,
                                       per_word_topics=True)
# Print the Keyword in the 10 topics
pprint(lda_model.print_topics())
doc_lda = lda_model[corpus]

2022-12-21 09:27:29,569 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric alpha at 0.1
2022-12-21 09:27:29,570 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric eta at 0.1
2022-12-21 09:27:29,571 | INFO | ldamodel.py:481 | __init__ | using serial LDA version on this node
2022-12-21 09:27:29,572 | INFO | ldamulticore.py:238 | update | running online LDA training, 10 topics, 10 passes over the supplied corpus of 107 documents, updating every 700 documents, evaluating every ~107 documents, iterating 50x with a convergence threshold of 0.001000
2022-12-21 09:27:29,581 | INFO | ldamulticore.py:279 | update | training LDA model using 7 processes
2022-12-21 09:27:29,645 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:27:29,653 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:27:31,96

2022-12-21 09:27:32,038 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 4, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:27:32,053 | INFO | ldamodel.py:1171 | show_topics | topic #7 (0.100): 0.036*"make" + 0.025*"colleague" + 0.024*"salary" + 0.024*"time" + 0.024*"always" + 0.024*"agent" + 0.024*"try" + 0.013*"provide" + 0.013*"safe" + 0.013*"need"
2022-12-21 09:27:32,053 | INFO | ldamodel.py:1171 | show_topics | topic #5 (0.100): 0.054*"good" + 0.041*"teammate" + 0.025*"lead" + 0.025*"team" + 0.023*"salary" + 0.023*"family" + 0.023*"relationship" + 0.023*"coworker" + 0.023*"acceptable" + 0.023*"fresh"
2022-12-21 09:27:32,054 | INFO | ldamodel.py:1171 | show_topics | topic #4 (0.100): 0.090*"good" + 0.031*"team" + 0.027*"salary" + 0.025*"benefit" + 0.024*"lead" + 0.022*"management" + 0.019*"environment" + 0.018*"problem" + 0.017*"setup" + 0.017*"wfh"
2022-12-21 09:27:32,054 | INFO | ldamodel.py:1171 | show_topics | topic #0 (0.100): 0.050

2022-12-21 09:27:32,141 | INFO | ldamodel.py:1171 | show_topics | topic #2 (0.100): 0.093*"good" + 0.029*"accommodate" + 0.029*"friendly" + 0.019*"management" + 0.019*"lead" + 0.019*"work" + 0.019*"employee" + 0.019*"approachable" + 0.019*"training" + 0.019*"team"
2022-12-21 09:27:32,142 | INFO | ldamodel.py:1171 | show_topics | topic #5 (0.100): 0.048*"good" + 0.044*"teammate" + 0.026*"lead" + 0.025*"team" + 0.023*"family" + 0.023*"salary" + 0.023*"relationship" + 0.023*"coworker" + 0.023*"fresh" + 0.023*"acceptable"
2022-12-21 09:27:32,142 | INFO | ldamodel.py:1049 | do_mstep | topic diff=0.014124, rho=0.315127
2022-12-21 09:27:32,145 | INFO | ldamodel.py:822 | log_perplexity | -6.216 per-word bound, 74.3 perplexity estimate based on a held-out corpus of 7 documents with 46 words
2022-12-21 09:27:32,145 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:27:32,146 | INFO | ldamulticore.py:294

[(0,
  '0.056*"salary" + 0.020*"supportive" + 0.020*"tl" + 0.020*"show" + '
  '0.020*"stay" + 0.020*"last" + 0.020*"resign" + 0.020*"heavy" + '
  '0.020*"wonderful" + 0.020*"overall"'),
 (1,
  '0.031*"good" + 0.028*"positive" + 0.028*"help" + 0.028*"easily" + '
  '0.025*"working" + 0.019*"opportunity" + 0.019*"give" + 0.019*"wfh" + '
  '0.019*"really" + 0.019*"offer"'),
 (2,
  '0.093*"good" + 0.029*"accommodate" + 0.029*"friendly" + 0.019*"management" '
  '+ 0.019*"lead" + 0.019*"work" + 0.019*"employee" + 0.019*"approachable" + '
  '0.019*"training" + 0.019*"team"'),
 (3,
  '0.121*"good" + 0.061*"trainer" + 0.038*"people" + 0.037*"kind" + '
  '0.033*"support" + 0.025*"benefit" + 0.025*"higherup" + 0.017*"considerate" '
  '+ 0.017*"company" + 0.013*"even"'),
 (4,
  '0.085*"good" + 0.032*"team" + 0.027*"salary" + 0.026*"benefit" + '
  '0.026*"lead" + 0.020*"management" + 0.020*"environment" + 0.020*"problem" + '
  '0.014*"setup" + 0.014*"wfh"'),
 (5,
  '0.047*"good" + 0.044*"teammate" +

> 4) Select optimal number of topics

In [17]:
# supporting function
def compute_coherence_values(corpus, dictionary, k, a, b):
    
    lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                           id2word=dictionary,
                                           num_topics=k, 
                                           random_state=100,
                                           chunksize=100,
                                           passes=10,
                                           alpha=a,
                                           eta=b)
    
    coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
    
    return coherence_model_lda.get_coherence()

In [18]:
# Baseline coherence score
from gensim.models import CoherenceModel
# Compute Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nBaseline Coherence Score: ', coherence_lda)

2022-12-21 09:27:32,224 | INFO | probability_estimation.py:155 | p_boolean_sliding_window | using ParallelWordOccurrenceAccumulator(processes=7, batch_size=64) to estimate probabilities from sliding windows
2022-12-21 09:27:34,046 | INFO | text_analysis.py:530 | terminate_workers | 7 accumulators retrieved from output queue
2022-12-21 09:27:34,076 | INFO | text_analysis.py:552 | merge_accumulators | accumulated word occurrence stats for 107 virtual documents



Baseline Coherence Score:  0.3840748861808779


In [19]:
# Optimize coherence score
import tqdm
grid = {}
grid['Validation_Set'] = {}
# Topics range
min_topics = 2
max_topics = 11
step_size = 1
topics_range = range(min_topics, max_topics, step_size)

# Alpha parameter
alpha = list(np.arange(0.01, 1, 0.3))
alpha.append('symmetric')
alpha.append('asymmetric')

# Beta parameter
beta = list(np.arange(0.01, 1, 0.3))
beta.append('symmetric')

# Validation sets
num_of_docs = len(corpus)
corpus_sets = [
#     gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.25)), 
#                gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.5)), 
               gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.75)), 
               corpus]

corpus_title = [
    #'25% Corpus','50% Corpus',
    '75% Corpus', '100% Corpus']

model_results = {'Validation_Set': [],
                 'Topics': [],
                 'Alpha': [],
                 'Beta': [],
                 'Coherence': []
                }

In [20]:
# # iterate through validation corpuses
# for i in tqdm_notebook(range(len(corpus_sets))):
#     # iterate through number of topics
#     for k in topics_range:
#         # iterate through alpha values
#         for a in alpha:
#             # iterare through beta values
#             for b in beta:
#                 # get the coherence score for the given parameters
#                 cv = compute_coherence_values(corpus=corpus_sets[i], dictionary=id2word, 
#                                               k=k, a=a, b=b)
#                 # Save the model results
#                 model_results['Validation_Set'].append(corpus_title[i])
#                 model_results['Topics'].append(k)
#                 model_results['Alpha'].append(a)
#                 model_results['Beta'].append(b)
#                 model_results['Coherence'].append(cv)


In [21]:
# # Create coherence dataframe
# coherence_results = (pd.DataFrame(model_results)
#                      .sort_values(by = 'Coherence')
#                     )

In [22]:
# # Plot results
# plot_df = (coherence_results
#              #.query("(Alpha != 'asymmetric') & (Alpha != 'symmetric') ", engine = 'python')
#              #.query("(Validation_Set != '25% Corpus') & (Validation_Set != '50% Corpus')", engine = 'python')
#              .query("(Alpha == 0.61) & (Beta == 0.61)", engine = 'python')
#              .sort_values(by = 'Topics', ascending = False)             
#             )
# plot_df

# # Plot
# plt.plot(plot_df['Topics'].to_list(), plot_df['Coherence'].to_list())
# plt.show()

In [23]:
# # Select optimal alpha & beta
# (coherence_results
#  .query("Topics == 5")
#  .sort_values(by = 'Coherence', ascending = False)
# )

> 5) Train final model with optimized hyperparameters

In [24]:
# Train final model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=5, 
                                           random_state=3,
                                           chunksize=100,
                                           passes=10,
                                           alpha='symmetric',
                                           eta='symmetric')

2022-12-21 09:27:34,432 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric alpha at 0.2
2022-12-21 09:27:34,433 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric eta at 0.2
2022-12-21 09:27:34,434 | INFO | ldamodel.py:481 | __init__ | using serial LDA version on this node
2022-12-21 09:27:34,435 | INFO | ldamulticore.py:238 | update | running online LDA training, 5 topics, 10 passes over the supplied corpus of 107 documents, updating every 700 documents, evaluating every ~107 documents, iterating 50x with a convergence threshold of 0.001000
2022-12-21 09:27:34,436 | INFO | ldamulticore.py:279 | update | training LDA model using 7 processes
2022-12-21 09:27:34,493 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:27:34,498 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:27:36,397

2022-12-21 09:27:36,478 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 4, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:27:36,495 | INFO | ldamodel.py:1171 | show_topics | topic #0 (0.200): 0.022*"supportive" + 0.019*"colleague" + 0.019*"agent" + 0.019*"great" + 0.019*"experience" + 0.019*"develop" + 0.013*"salary" + 0.013*"provide" + 0.013*"help" + 0.013*"skill"
2022-12-21 09:27:36,496 | INFO | ldamodel.py:1171 | show_topics | topic #1 (0.200): 0.025*"good" + 0.021*"wfh" + 0.020*"working" + 0.020*"positive" + 0.020*"provide" + 0.019*"colleague" + 0.016*"opportunity" + 0.016*"atmosphere" + 0.016*"give" + 0.016*"help"
2022-12-21 09:27:36,496 | INFO | ldamodel.py:1171 | show_topics | topic #2 (0.200): 0.076*"good" + 0.034*"company" + 0.020*"work" + 0.020*"team" + 0.015*"approachable" + 0.015*"management" + 0.015*"especially" + 0.015*"lead" + 0.015*"salary" + 0.015*"account"
2022-12-21 09:27:36,497 | INFO | ldamodel.py:1171 | show_topics | 

2022-12-21 09:27:36,593 | INFO | ldamodel.py:1171 | show_topics | topic #3 (0.200): 0.142*"good" + 0.047*"company" + 0.030*"salary" + 0.026*"trainer" + 0.017*"people" + 0.015*"management" + 0.014*"kind" + 0.014*"experience" + 0.014*"accommodate" + 0.014*"benefit"
2022-12-21 09:27:36,593 | INFO | ldamodel.py:1171 | show_topics | topic #4 (0.200): 0.085*"good" + 0.028*"team" + 0.026*"benefit" + 0.022*"wfh" + 0.022*"friendly" + 0.019*"environment" + 0.019*"setup" + 0.016*"colleague" + 0.016*"lead" + 0.015*"management"
2022-12-21 09:27:36,594 | INFO | ldamodel.py:1049 | do_mstep | topic diff=0.007144, rho=0.315127
2022-12-21 09:27:36,597 | INFO | ldamodel.py:822 | log_perplexity | -5.872 per-word bound, 58.6 perplexity estimate based on a held-out corpus of 7 documents with 46 words
2022-12-21 09:27:36,597 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:27:36,598 | INFO | ldamulticore.py:294 | 

> 4) Visualizing topics

For (1), you can manually select each topic to view its top most frequent and/or “relevant” terms, using different values of the λ parameter. This can help when you’re trying to assign a human interpretable name or “meaning” to each topic.

For (2), exploring the Intertopic Distance Plot can help you learn about how topics relate to each other, including potential higher-level structure between groups of topics.

**Interpretation:**


*Bar chart*
- Each bubble represents a topic. The larger the bubble, the higher percentage of the number of responses in the corpus is about that topic.
- Blue bars represent the overall frequency of each word in the corpus. If no topic is selected, the blue bars of the most frequently used words will be displayed.
- Red bars give the estimated number of times a given term was generated by a given topic. The word with the longest red bar is the word that is used the most by the responses belonging to that topic.
- <u>*Lambda*</u> -- Decreasing the lambda parameter, increases the weight of the ratio of the frequency of word given the topic / Overall frequency of the word in the documents. Important words for the given topic moves upward.
    - Lambda = 1, top words by frequency
    - Lambda = 0, top words by relevance to topic

*Bubble plot*
- The further the bubbles are away from each other, the more different they are. 
- The larger the bubble, the more frequent is the topic in the documents.
- Distance between the topics is an approximation of semantic relationship between the topics.
- The topic which shares common words will be overlapping (closer in distance) in comparison to the non-overlapping topic.
- A good topic model will have big and non-overlapping bubbles scattered throughout the chart. As we can see from the graph, the bubbles are clustered within one place.
- A topic model with a low number of topics will have big non-overlapping bubbles, scattered throughout the chart whereas, the topic model with a high number of topics, will have many overlapping small size bubbles, clustered in the chart.


In [25]:
import pyLDAvis.gensim
import pickle 
import pyLDAvis

In [26]:
# Visualize the topics
pyLDAvis.enable_notebook()
LDAvis_prepared = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
LDAvis_prepared = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
LDAvis_prepared

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4     -0.049036  0.118970       1        1  29.478808
3     -0.093957 -0.087850       2        1  20.196218
2     -0.044731 -0.010032       3        1  18.963229
1      0.056419  0.001336       4        1  18.069087
0      0.131305 -0.022424       5        1  13.292659, topic_info=          Term       Freq      Total Category  logprob  loglift
22        good  61.000000  61.000000  Default  30.0000  30.0000
115    company  15.000000  15.000000  Default  29.0000  29.0000
21   colleague   9.000000   9.000000  Default  28.0000  28.0000
19     benefit   8.000000   8.000000  Default  27.0000  27.0000
13    positive   5.000000   5.000000  Default  26.0000  26.0000
..         ...        ...        ...      ...      ...      ...
155       also   0.789196   2.087077   Topic5  -4.9426   1.0455
123         tl   0.789065   2.817413   Topic5  -4.9428   0.7452
158       much   0.788922   2.142705   Topic5  -4.9430   1.0188
157  knowledge   0.788755   2.103519   Topic5  -4.9432   1.0370
218  different   0.788691   2.833819   Topic5  -4.9433   0.7390

[315 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
163       2  0.462923         able
163       4  0.462923         able
58        1  0.302648  accommodate
58        2  0.302648  accommodate
58        3  0.302648  accommodate
...     ...       ...          ...
18        5  0.105518      working
150       1  0.674324     workmate
150       3  0.337162     workmate
84        2  0.699328         year
246       2  0.699329      yearend

[373 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[5, 4, 3, 2, 1])

# Post-modeling Exploratory Data Analysis

*Assign respondents to their topic clusters*

In [27]:
# Topic distribution for the whole document. Each element in the list is a pair of a topic’s id, and the probability that was assigned to it.
topic_dist = list(lda_model.get_document_topics(corpus))


# Classify responses to topics
cluster = []
probability = []

for i in range(0, len(topic_dist)):
    clust = [x for x,y in topic_dist[i]]
    prob = [y for x,y in topic_dist[i]]
    get_index = prob.index(max(prob))
    max_val = topic_dist[i][get_index]
    cluster.append(max_val[0])
    probability.append(max_val[1])
    
# Add classification to df
analysis_df['Topic'] = cluster
analysis_df['Topic_Probability'] = probability

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [28]:
# Map topics to Oracle ID
id_topic_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['Topic']))
id_prob_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['Topic_Probability']))
id_rate_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']))

# For those with missing Oracle ID, map to names
name_mapper = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','Topic']]
                
                     .set_index("Name").to_dict()['Topic']
)

name_mapper2 = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','Topic_Probability']]
                
                     .set_index("Name").to_dict()['Topic_Probability']
)

rating_mapper = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']]
                
                     .set_index("Name").to_dict()['How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']
)

# Filter attrition df to contain respondents only
respondents_demog_df = (attrition_df
                         .assign(Topic1 = lambda x: x['EE Oracle Id'].map(id_topic_mapper),
                                 Prob1 = lambda x: x['EE Oracle Id'].map(id_prob_mapper),
                                 Rate1 = lambda x: x['EE Oracle Id'].map(id_rate_mapper),
                                 Name = lambda x: (x['First Name'] + " " + x['Last Name']).apply(lambda x: x.upper()),
                                 Topic = lambda x: np.where(x['Topic1'].isna(), x['Name'].map(name_mapper), x['Topic1']),
                                 Topic_Probability = lambda x: np.where(x['Prob1'].isna(), x['Name'].map(name_mapper2), x['Prob1']),
                                Rating = lambda x: np.where(x['Rate1'].isna(), x['Name'].map(rating_mapper), x['Rate1']),

                         )
                         .drop(['Topic1','Prob1','Rate1'], axis = 1)
                         .query("Topic.notna()", engine = 'python')
                         .sort_values(by = 'Topic')
                        )

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


> 1) How many respondents belong to each topic cluster & what is the average probability score?

In [29]:
# "Number of Employees per Cluster & Average Probability Score")
gen_stats = (analysis_df
             .groupby(['Topic']).agg({'Oracle ID of former employee':'nunique', 'Topic_Probability':'describe'})
             .reset_index()
            )

gen_stats

print("Number of Employees per Cluster & Average Probability Score")


/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Topic Oracle ID of former employee Topic_Probability                      \
        Oracle ID of former employee             count      mean       std   
0     0                           14              14.0  0.848610  0.085870   
1     1                           12              12.0  0.900448  0.079333   
2     2                           18              18.0  0.899918  0.044122   
3     3                           29              29.0  0.832679  0.094520   
4     4                           34              34.0  0.853108  0.107096   

                                                     
        min       25%       50%       75%       max  
0  0.726173  0.798131  0.851420  0.926517  0.963858  
1  0.730932  0.858339  0.943472  0.954937  0.959597  
2  0.795741  0.869334  0.914387  0.932296  0.949385  
3  0.595515  0.798694  0.838868  0.899212  0.952415  
4  0.543924  0.837351  0.875525  0.923961  0.965034

Number of Employees per Cluster & Average Probability Score


> 2) Tenure, Topic Probability, and Employee Count by By Attrition Stage

In [30]:
(respondents_demog_df
 .groupby(['Topic','Stage Attrition']).agg({'EE Oracle Id':'nunique','Tenure In Months':['mean', 'std','median'],'Topic_Probability':['mean', 'std','median']})
 .unstack('Stage Attrition')
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


EE Oracle Id            Tenure In Months             \
                       nunique                        mean              
Stage Attrition Pre Production Production   Pre Production Production   
Topic                                                                   
0.0                          2         12             17.5  22.416667   
1.0                          2         10              2.5  20.600000   
2.0                          4         14              1.0  23.285714   
3.0                          4         25              7.0  24.520000   
4.0                          2         32              1.5  30.281250   

                                                                     \
                           std                    median              
Stage Attrition Pre Production Production Pre Production Production   
Topic                                                                 
0.0                  17.677670  13.833151           17.5       20.5   
1.0                   2.121320  12.980327            2.5       19.5   
2.0                   0.000000  19.177024            1.0       20.0   
3.0                   9.345231  20.746727            2.5       16.0   
4.0                   0.707107  26.961423            1.5       23.0   

                Topic_Probability                                       \
                             mean                       std              
Stage Attrition    Pre Production Production Pre Production Production   
Topic                                                                    
0.0                      0.841658   0.849769       0.061594   0.091429   
1.0                      0.957244   0.889089       0.003327   0.082649   
2.0                      0.876719   0.906547       0.064522   0.037042   
3.0                      0.841697   0.831236       0.096024   0.096201   
4.0                      0.849361   0.853342       0.017359   0.110448   

                                           
                        median             
Stage Attrition Pre Production Production  
Topic                                      
0.0                   0.841658   0.851420  
1.0                   0.957244   0.931711  
2.0                   0.880876   0.921996  
3.0                   0.846161   0.838868  
4.0                   0.849361   0.891942

> 3) Tenure, Topic Probability, and Employee Count by By Job Name

In [31]:
(respondents_demog_df
 .groupby(['Topic','Job Name','Stage Attrition']).agg({'EE Oracle Id':'nunique','Tenure In Months':['mean', 'std','median']})
 .unstack('Job Name')
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


EE Oracle Id                                        \
                           nunique                                         
Job Name                     CSR I CSR II CSR III Chat Sales Associate I   
Topic Stage Attrition                                                      
0.0   Pre Production           1.0    1.0     NaN                    NaN   
      Production               2.0    7.0     1.0                    NaN   
1.0   Pre Production           1.0    1.0     NaN                    NaN   
      Production               2.0    7.0     NaN                    1.0   
2.0   Pre Production           1.0    3.0     NaN                    NaN   
      Production               NaN   10.0     4.0                    NaN   
3.0   Pre Production           2.0    2.0     NaN                    NaN   
      Production               4.0   15.0     3.0                    NaN   
4.0   Pre Production           1.0    1.0     NaN                    NaN   
      Production               5.0   22.0     5.0                    NaN   

                                          Tenure In Months             \
                                                      mean              
Job Name              ISR I TSR I TSR III            CSR I     CSR II   
Topic Stage Attrition                                                   
0.0   Pre Production    NaN   NaN     NaN             5.00  30.000000   
      Production        NaN   NaN     2.0            22.00  18.571429   
1.0   Pre Production    NaN   NaN     NaN             4.00   1.000000   
      Production        NaN   NaN     NaN            26.50  16.142857   
2.0   Pre Production    NaN   NaN     NaN             1.00   1.000000   
      Production        NaN   NaN     NaN              NaN  22.800000   
3.0   Pre Production    NaN   NaN     NaN             2.50  11.500000   
      Production        1.0   1.0     1.0            42.25  18.000000   
4.0   Pre Production    NaN   NaN     NaN             2.00   1.000000   
      Production        NaN   NaN     NaN            28.00  31.863636   

                                                                             \
                                                                              
Job Name                 CSR III Chat Sales Associate I ISR I TSR I TSR III   
Topic Stage Attrition                                                         
0.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       50.000000                    NaN   NaN   NaN    22.5   
1.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production             NaN                   40.0   NaN   NaN     NaN   
2.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       24.500000                    NaN   NaN   NaN     NaN   
3.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       15.333333                    NaN  59.0  56.0    13.0   
4.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       25.600000                    NaN   NaN   NaN     NaN   

                                                                               \
                             std                                                
Job Name                   CSR I     CSR II    CSR III Chat Sales Associate I   
Topic Stage Attrition                                                           
0.0   Pre Production         NaN        NaN        NaN                    NaN   
      Production        7.071068  13.721724        NaN                    NaN   
1.0   Pre Production         NaN        NaN        NaN                    NaN   
      Production       20.506097   9.227289        NaN                    NaN   
2.0   Pre Production         NaN   0.000000        NaN                    NaN   
      Production             NaN  21.852028  12.583057                    NaN   
3.0   Pre Product

> 4) Tenure, Topic Probability, and Employee Count by By Job Name

In [32]:
(respondents_demog_df
 .groupby(['Topic','Stage Attrition']).agg({'Rating': ['mean', 'std','median']})
 .round(1)
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Rating            
                        mean  std median
Topic Stage Attrition                   
0.0   Pre Production     3.5  0.7    3.5
      Production         3.6  1.1    4.0
1.0   Pre Production     5.0  NaN    5.0
      Production         4.2  1.2    5.0
2.0   Pre Production     3.7  1.2    3.0
      Production         4.2  0.6    4.0
3.0   Pre Production     4.7  0.6    5.0
      Production         4.2  0.5    4.0
4.0   Pre Production     4.5  0.7    4.5
      Production         4.3  0.7    4.0